In [1]:
!pip install pandas

In [6]:
import json
import os

with open("../outputs/resume_text.txt", "r", encoding="utf-8") as f:
    resume_text = f.read()

with open("../outputs/skills.json", "r", encoding="utf-8") as f:
    skills_data = json.load(f)

technical_skills_list = skills_data["llm_based"].get("technical_skills", skills_data["regex_based"]["skills"]["technical"])
skills_data_compat = {"skills": {"technical": technical_skills_list, "soft": skills_data["regex_based"]["skills"]["soft"]}}
skills_data = skills_data_compat

print("Loaded resume text and skills data")
print(f"Technical skills found: {len(skills_data['skills']['technical'])}")
print(f"Soft skills found: {len(skills_data['skills']['soft'])}")

Loaded resume text and skills data
Technical skills found: 13
Soft skills found: 8


In [3]:
def calculate_ats_score(text, skills_data):
    score = 0
    feedback = []

    # Contact info (20 points)
    import re
    has_email = bool(re.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', text))
    has_phone = bool(re.search(r'\d{10}', text))

    if has_email:
        score += 10
    else:
        feedback.append("Add a professional email address")

    if has_phone:
        score += 10
    else:
        feedback.append("Add a phone number")

    # Skills (30 points)
    technical_count = len(skills_data["skills"]["technical"])
    if technical_count >= 8:
        score += 30
    elif technical_count >= 5:
        score += 20
        feedback.append("Add more relevant technical skills")
    elif technical_count >= 1:
        score += 10
        feedback.append("Your technical skills section is thin, add more")
    else:
        feedback.append("No technical skills found")

    # Section keywords (30 points)
    section_keywords = ["experience", "education", "project", "skill", "certification", "achievement"]
    text_lower = text.lower()
    found_sections = [kw for kw in section_keywords if kw in text_lower]
    section_score = min(len(found_sections) * 5, 30)
    score += section_score
    if len(found_sections) < 4:
        missing = [kw for kw in section_keywords if kw not in found_sections]
        feedback.append(f"Consider adding sections like: {', '.join(missing)}")

    # Length check (20 points)
    word_count = len(text.split())
    if word_count >= 150:
        score += 20
    elif word_count >= 80:
        score += 10
        feedback.append("Resume content seems short")
    else:
        feedback.append("Resume is too short")

    score = min(score, 100)
    if not feedback:
        feedback.append("Great! Your resume covers the key ATS-friendly elements")

    return {"ats_score": score, "feedback": feedback}


def calculate_readiness_score(ats_score, dsa_score=0, projects_score=0, communication_score=0, interview_score=0, consistency_score=0):
    weights = {"resume": 0.20, "dsa": 0.20, "projects": 0.20, "communication": 0.15, "interview": 0.15, "consistency": 0.10}

    total = (
        ats_score * weights["resume"] +
        dsa_score * weights["dsa"] +
        projects_score * weights["projects"] +
        communication_score * weights["communication"] +
        interview_score * weights["interview"] +
        consistency_score * weights["consistency"]
    )

    breakdown = {
        "resume": round(ats_score, 2),
        "dsa": round(dsa_score, 2),
        "projects": round(projects_score, 2),
        "communication": round(communication_score, 2),
        "interview": round(interview_score, 2),
        "consistency": round(consistency_score, 2)
    }

    weak_areas = [k for k, v in breakdown.items() if v < 50]

    return {"career_readiness_score": round(total, 2), "breakdown": breakdown, "weak_areas": weak_areas}

print("Career Readiness Agent defined")

Career Readiness Agent defined


In [4]:
# Estimate a rough projects_score based on number of projects found
projects_count = len(skills_data.get("projects", []))
estimated_projects_score = min(projects_count * 20, 100)

ats_result = calculate_ats_score(resume_text, skills_data)
readiness_result = calculate_readiness_score(
    ats_score=ats_result["ats_score"],
    projects_score=estimated_projects_score
)

print("Career Readiness Agent: SUCCESS")
print(f"\nATS Score: {ats_result['ats_score']}/100")
print(f"Feedback: {ats_result['feedback']}")
print(f"\nCareer Readiness Score: {readiness_result['career_readiness_score']}/100")
print(f"Breakdown: {json.dumps(readiness_result['breakdown'], indent=2)}")
print(f"Weak areas: {readiness_result['weak_areas']}")

Career Readiness Agent: SUCCESS

ATS Score: 95/100
Feedback: ['Great! Your resume covers the key ATS-friendly elements']

Career Readiness Score: 39.0/100
Breakdown: {
  "resume": 95,
  "dsa": 0,
  "projects": 100,
  "communication": 0,
  "interview": 0,
  "consistency": 0
}
Weak areas: ['dsa', 'communication', 'interview', 'consistency']


In [5]:
final_result = {
    "ats": ats_result,
    "readiness": readiness_result
}

with open("../outputs/career_readiness.json", "w", encoding="utf-8") as f:
    json.dump(final_result, f, indent=2)

print("Career readiness data saved to ../outputs/career_readiness.json")
print("Notebook 3 (Career Readiness Agent) — COMPLETE")

Career readiness data saved to ../outputs/career_readiness.json
Notebook 3 (Career Readiness Agent) — COMPLETE
